In [ ]:
dbutils.widgets.text("catalog_name", "", "Catalog name")
dbutils.widgets.text("schema_name", "", "Schema name")

Create the catalog and schema if not already present

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

query = f"""
CREATE CATALOG IF NOT EXISTS {catalog_name};
"""

spark.sql(query)

query = f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name};
"""

spark.sql(query)

In [ ]:
query = f"""
USE CATALOG {catalog_name};
"""

spark.sql(query)

query = f"""
USE SCHEMA {schema_name};
"""

spark.sql(query)

Create or replace the elt configuration table

In [ ]:
create_query = f"""
CREATE OR REPLACE TABLE etl_meta_config
(emc_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY COMMENT "Table group id primary key to identify a single row",
  emc_task_id STRING NOT NULL COMMENT "Task string identifier",
  emc_seq STRING NOT NULL COMMENT "sequence number. Will determine the execution order for the rows with the same emc_task_id.",
  emc_target_layer STRING NOT NULL COMMENT "Medallion layer Bronze, Silver or Gold", 
  emc_func_type STRING NOT NULL COMMENT "Function of this particular entry",
  emc_func_dtls VARIANT NOT NULL COMMENT "Details related to the func_type",
  emc_target STRING COMMENT "Target. Table or view name.May be empty",
  emc_out_mode STRING COMMENT "Append or overwrite modes",
  emc_table_retention INT COMMENT "Retention in months",
  emc_target_storage_loc STRING COMMENT "Storage location ADLS or S3. Leave blank for a managed table",
  emc_active STRING NOT NULL COMMENT "Y or N. Only active rows will be processed",
  emc_desc STRING COMMENT "Free text description"
  )
  CLUSTER BY (emc_task_id, emc_target_layer, emc_func_type, emc_active)  -- or: CLUSTER BY AUTO
  COMMENT "Configuration for workflow meta transformation" 
"""

spark.sql(create_query)

spark.sql("""
    ALTER TABLE etl_meta_config 
    ADD CONSTRAINT target_layer_chk 
    CHECK (emc_target_layer IN ('BRONZE', 'SILVER', 'GOLD'))
""")

spark.sql("""
    ALTER TABLE etl_meta_config 
    ADD CONSTRAINT function_type_chk 
    CHECK (emc_func_type IN ('SQLLOAD', 'NOTEBOOK', 'SQL'))
""")

spark.sql("""
    ALTER TABLE etl_meta_config 
    ADD CONSTRAINT out_mode_chk 
    CHECK (emc_out_mode IN ('', 'APPEND', 'OVERWRITE'))
""")

spark.sql("""
    ALTER TABLE etl_meta_config 
    ADD CONSTRAINT active_chk 
    CHECK (emc_active IN ('Y', 'N'))
""")

Create a logging table

In [ ]:
createlog_query = """
  CREATE OR REPLACE TABLE etl_table_load_status
  (etls_log_id STRING COMMENT "log id primary key to identify a single row",
    etls_load_date DATE NOT NULL COMMENT "",
    etls_catalog STRING COMMENT "",
    etls_schema STRING COMMENT "",
    etls_table_name STRING NOT NULL COMMENT "",
    etls_prev_ver BIGINT COMMENT "",
    etls_new_ver BIGINT COMMENT "",
    etls_rows_inserted INT COMMENT "",
    etls_rows_updated INT COMMENT "",
    etls_rows_deleted INT COMMENT "",
    etls_jobname STRING COMMENT "",
    etls_jobrunid STRING COMMENT "",
    etls_jobid STRING COMMENT "",
    etls_taskrunid STRING COMMENT "",
    etls_taskname STRING COMMENT "",
    etls_notebook STRING COMMENT "",
    etls_run_by STRING COMMENT "",
    etls_end_timestamp TIMESTAMP COMMENT "",
    etls_start_timestamp TIMESTAMP COMMENT "",
    etls_status STRING NOT NULL COMMENT "FAILED OR SUCCESS"
  )
  CLUSTER BY (etls_catalog, etls_schema, etls_table_name, etls_status)  -- or: CLUSTER BY AUTO
  TBLPROPERTIES (
  -- Auto Optimize family
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true')
  COMMENT "Log table for table load status"
"""

spark.sql(createlog_query)